# Ana eğitim (ChangeMamba-MMBDA Tiny)

İnce defter: bütün mantık `src/hasar` paketinde. "Run All" ile baştan sona çalışır; kopmadan sonra yeniden
"Run All" aynı sonucu verir (her hücre yapılmış işi atlar). Ayrıntı: `docs/tasarim.md` Bölüm 8, `docs/colab.md`.

**Önce yalnız şu iki satırı doldurun:** `REF` (etiket ya da commit) ve `STEPS_OVERRIDE` (prova için; asıl eğitimde `None`).
Türkiye değerlendirmesi (hücre 6) varsayılan olarak kapalıdır: `RUN_FINAL = True` yalnız Faz 6'da, KN1 ve
`varyant-kilidi` sonrasında, `REF = "varyant-kilidi"` (ya da ondan sonraki bir commit) ile yapılır. Varyant (Faz 8)
için aynı defter `CONFIG = "configs/varyant.yaml"` ve `NAME = "varyant"` ile çalışır.

In [ ]:
# Hücre 1: Drive, etiketli commit, kurulum, sırlar
import os
import subprocess
from pathlib import Path

from google.colab import drive, userdata

REF = "protokol-kilidi"  # klonlanacak etiket veya commit; final için "varyant-kilidi"
STEPS_OVERRIDE = None  # ör. 2000 (kısa prova); asıl eğitimde None
RUN_FINAL = False  # Faz 6'da, KN1 ve varyant-kilidi sonrasında True
CONFIG = "configs/ana.yaml"  # varyant için "configs/varyant.yaml"
NAME = "ana"  # sonuç klasörü adı; varyant için "varyant"

drive.mount("/content/drive")
REPO = Path("/content/deprem-hasar-haritasi")
DRIVE = Path("/content/drive/MyDrive/deprem-hasar")
DATA = Path("/content/data/bright")
CKPT = Path("/content/data/checkpoints")


def sh(cmd):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)


if not REPO.exists():
    sh(f"git clone https://github.com/OgzhnTarhn/deprem-hasar-haritasi.git {REPO}")
sh(f"cd {REPO} && git fetch --tags && git checkout --quiet {REF}")
dirty = subprocess.run(f"cd {REPO} && git status --porcelain", shell=True, capture_output=True, text=True).stdout
assert not dirty.strip(), "Çalışma ağacı kirli; etiketli commit değiştirilmemeli."
COMMIT = subprocess.run(f"cd {REPO} && git rev-parse HEAD", shell=True, capture_output=True, text=True).stdout.strip()
print("commit:", COMMIT)
# Colab'ın torch ve timm sürümleri kullanılır; torch yeniden kurulmaz.
sh(f"pip install -q -e {REPO} einops")
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    print("HF_TOKEN sırrı yok; indirme anonim yapılır.")
DRIVE.mkdir(parents=True, exist_ok=True)

In [ ]:
# Hücre 2: Ortam kapısı ve selective_scan çekirdeği (en çok 1 saat; geçmezse DamageFormer yedeğine geçilir)
import shutil
import time

import torch

props = torch.cuda.get_device_properties(0)
print(props.name, round(props.total_memory / 2**30, 1), "GB")
# Colab A100 seçilse de bazen L4 (22 GB) veriyor; ikisi de bf16 destekler. Bellek ve CU kapı 5 ve 8'de ölçülür.
assert any(gpu in props.name for gpu in ("A100", "L4")), f"A100 ya da L4 gerekli, gelen: {props.name}"
free_gb = shutil.disk_usage("/content").free / 2**30
assert free_gb >= 60, f"Disk yetersiz: {free_gb:.0f} GB"
print("torch", torch.__version__)

CM = Path("/content/ChangeMamba")
CM_COMMIT = "9ce9cec13f9ea14bc0ad91f071577ec9b3a97983"
try:
    import selective_scan_cuda_oflex  # noqa: F401

    print("çekirdek zaten kurulu")
except ImportError:
    if not CM.exists():
        sh(f"git clone https://github.com/ChenHongruixuan/ChangeMamba.git {CM}")
    sh(f"cd {CM} && git checkout --quiet {CM_COMMIT}")
    t0 = time.time()
    try:
        subprocess.run(f"cd {CM}/kernels/selective_scan && pip install -q .", shell=True, check=True, timeout=3600)
        print(f"çekirdek derlendi ({(time.time() - t0) / 60:.0f} dk)")
    except Exception as exc:
        print("ÇEKİRDEK DERLENEMEDİ, eğitim DamageFormer yedeğiyle yapılacak:", exc)

In [ ]:
# Hücre 3: Veri (VM diskine) ve ön eğitim ağırlığı
sh(f"cd {REPO} && hasar indir --root {DATA} --cache /content/data/cache")  # .done varsa atlanır
sh(f"cd {REPO} && hasar agirlik --out {CKPT}")  # yalnız ImageNet vssm_tiny (md5 denetimli)
PRETRAINED = CKPT / "vssm_tiny_0230_ckpt_epoch_262.pth"

In [ ]:
# Hücre 4: Yeniden üretim (kapı 1) ve ön kontrol kapıları -> preflight.json
REPRO = REPO / "results" / "yeniden-uretim"
for key in ["siamcrnn", "unet", "damageformer"]:
    if not (REPRO / f"{key}_test.json").exists():
        sh(f"cd {REPO} && hasar yeniden-uret --model {key} --data {DATA} --out {REPRO} --cache {CKPT} --device cuda")
OUT = DRIVE / "kosular"
steps = f"--steps {STEPS_OVERRIDE}" if STEPS_OVERRIDE else ""
if RUN_FINAL:
    print("RUN_FINAL=True: ön kontrol atlandı (eğitim bitmiş olmalı).")
else:
    rc = subprocess.run(
        f"cd {REPO} && hasar on-kontrol --config {CONFIG} --data {DATA} --out {OUT} "
        f"--device cuda --pretrained {PRETRAINED} --repro-dir {REPRO}",
        shell=True,
    ).returncode
    assert rc == 0, "Ön kontrol kapılarından biri geçmedi; preflight.json'a bakın. Eğitim başlamaz."

In [ ]:
# Hücre 5: Ana eğitim (last.pt varsa tam durumla devam eder, DONE varsa atlanır)
sh(
    f"cd {REPO} && hasar egit --config {CONFIG} --data {DATA} --out {OUT} "
    f"--device cuda --pretrained {PRETRAINED} --commit {COMMIT} {steps}"
)

In [ ]:
# Hücre 6: Türkiye'de tek değerlendirme (Faz 6/8). Eğitim DONE değilse, protokol-kilidi ve varyant-kilidi
# etiketleri HEAD'in atası değilse ya da çalışma ağacı kirliyse çalışmaz. Çıktılar depo dışına (Drive) yazılır,
# böylece ağaç temiz kalır; sonra yerelde results/<NAME>/ altına kopyalanıp commit'lenir.
if RUN_FINAL:
    RES = DRIVE / "sonuclar" / NAME
    INST = Path("/content/data/ornekler")
    CACHE = Path("/content/data/final_cache")
    sh(f"cd {REPO} && hasar ornekler --dest {INST} --cache /content/data/cache")  # yalnız sabit 1.001 kare
    if not (RES / "sicaklik.json").exists():  # sıcaklık yalnız 13 olayın doğrulamasında seçilir
        sh(f"cd {REPO} && hasar sicaklik --config {CONFIG} --data {DATA} --runs {OUT} --device cuda --out {RES}")
    if (RES / "turkiye.json").exists():
        print("turkiye.json var: Türkiye değerlendirmesi zaten yapıldı, ikinci kez çalışmaz.")
    else:
        sh(
            f"cd {REPO} && hasar final --config {CONFIG} --data {DATA} --runs {OUT} --device cuda "
            f"--out {RES} --cache {CACHE} --instances {INST}"
        )
    sh(f"mkdir -p {DRIVE / 'final_cache'} && cp -rn {CACHE}/. {DRIVE / 'final_cache'}/")
    print("Sonuçlar:", RES)
else:
    print("RUN_FINAL=False: Türkiye değerlendirmesi atlandı.")

In [ ]:
# Hücre 7: Kopyaları doğrula ve çalışma zamanını bırak (CU yakımı durur)
import json

from google.colab import runtime

done = list(OUT.glob("*/DONE"))
print("DONE:", [str(p) for p in done])
for p in done:
    print(json.loads(p.read_text(encoding="utf-8")))
runtime.unassign()